# NLP

NLP, ou Processamento de Linguagem Natural, e uma area da ciencia de dados que estuda como transformar textos em informacao estruturada. Com essas tecnicas, conseguimos analisar noticias, identificar temas, contar palavras, comparar documentos e preparar dados textuais para modelos de aprendizado de maquina.

In [1]:
import json
from pathlib import Path

import pandas as pd

noticias = []

for arquivo in sorted(Path("../WebScrapping/data/").glob("*.json")):
    with arquivo.open(encoding="utf-8") as f:
        noticias.append(json.load(f))

df = pd.DataFrame(noticias)

print(f"Antes: {len(df)} noticias")

df = df.drop_duplicates(subset="url").reset_index(drop=True)

print(f"Depois: {len(df)} noticias")

df.head()

Antes: 2000 noticias
Depois: 1997 noticias


,url,titulo,subtitulo,categoria,data_publicacao,imagem,corpo
0,https://www.cnnbrasil.com.br/esportes/futebol/...,Santos acerta renovação de Robinho Jr. até o f...,Clube da Baixada Santista define condições de ...,Santos,2026-04-14T16:18:41-03:00,https://admin.cnnbrasil.com.br/wp-content/uplo...,O Santos encaminhou a extensão de contrato do ...
1,https://www.cnnbrasil.com.br/esportes/futebol/...,Cruzeiro anuncia renovação com Artur Jorge apó...,Treinador iniciou no clube celeste em 22 de ma...,Cruzeiro,2026-04-14T15:52:24-03:00,https://admin.cnnbrasil.com.br/wp-content/uplo...,"O Cruzeiro anunciou, nesta terça-feira (14), a..."
2,https://www.cnnbrasil.com.br/esportes/futebol/...,Ex-Corinthians cita caso vivido em clube e cri...,Betão comentou o episódio recente no Dérbi pau...,Corinthians,2026-04-13T18:37:05-03:00,https://admin.cnnbrasil.com.br/wp-content/uplo...,Um episódio lamentável de racismo marcou o clá...
3,https://www.cnnbrasil.com.br/esportes/futebol/...,Cruzeiro: Pedro Junio explica escolha por Artu...,"Ao CNN Esportes S/A deste domingo (12), vice-p...",Cruzeiro,2026-04-10T09:00:53-03:00,https://admin.cnnbrasil.com.br/wp-content/uplo...,"Pedro Junio, vice-presidente do Cruzeiro, expl..."
4,https://www.cnnbrasil.com.br/esportes/futebol/...,"""Eu sou 200% Flamengo"", diz Leonardo Jardim ap...",Treinador destaca evolução da equipe e comenta...,Flamengo,2026-03-12T08:46:42-03:00,https://admin.cnnbrasil.com.br/wp-content/uplo...,Após a vitória do Flamengo sobre o Cruzeiro po...


## Trabalhando apenas com o texto

Por enquanto, vamos trabalhar somente com a coluna `texto`, que contem o corpo completo de cada noticia. As outras colunas, como `titulo`, `descricao`, `tags` e `data`, continuam no `DataFrame`, mas vamos deixar elas de lado neste primeiro momento para entender melhor o conteudo textual.

## Passos da analise

Vamos preparar os textos aos poucos:

1. Limpar os textos.
2. Remover palavras muito comuns.
3. Criar uma representacao Bag of Words.
4. Contar palavras frequentes.
5. Transformar textos em numeros para analises posteriores.

## 1. Limpeza basica

Na celula abaixo:

- `wordpunct_tokenize` separa o texto em palavras e pontuacao.
- `texto.lower()` coloca tudo em minusculas.
- `unidecode(texto)` troca letras acentuadas por letras sem acento.
- `token.isalnum()` mantem apenas letras e numeros.
- `" ".join(tokens)` junta os tokens em uma frase limpa.
- `.apply(limpar_texto)` aplica a funcao em todas as noticias.

Exemplo: `"Olá, Senado!"` vira `"ola senado"`.

In [5]:
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode


def limpar_texto(texto):
    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [token for token in tokens if token.isalnum()]
    return " ".join(tokens)


df["texto_limpo"] = df["corpo"].apply(limpar_texto)

df[["corpo", "texto_limpo"]].head()

,corpo,texto_limpo
0,O Santos encaminhou a extensão de contrato do ...,o santos encaminhou a extensao de contrato do ...
1,"O Cruzeiro anunciou, nesta terça-feira (14), a...",o cruzeiro anunciou nesta terca feira 14 a ren...
2,Um episódio lamentável de racismo marcou o clá...,um episodio lamentavel de racismo marcou o cla...
3,"Pedro Junio, vice-presidente do Cruzeiro, expl...",pedro junio vice presidente do cruzeiro explic...
4,Após a vitória do Flamengo sobre o Cruzeiro po...,apos a vitoria do flamengo sobre o cruzeiro po...


## 2. Removendo stopwords

Stopwords sao palavras muito comuns, como `a`, `o`, `de`, `para` e `que`. Elas aparecem muito, mas geralmente ajudam pouco a entender o tema de um texto.

Na celula abaixo:

- `stopwords.words("portuguese")` carrega stopwords em portugues.
- `texto.split()` separa o texto limpo em palavras.
- `token not in stopwords_pt` remove as palavras muito comuns.
- `.str.join(" ")` junta os tokens restantes em um texto sem stopwords.
- `.apply(remover_stopwords)` aplica a funcao em todas as noticias.

Exemplo: `"o senador falou com a imprensa"` vira `["senador", "falou", "imprensa"]`.

In [6]:
import nltk

from nltk.corpus import stopwords


nltk.download("stopwords", quiet=True)

stopwords_pt = stopwords.words("portuguese")
stopwords_pt = [unidecode(palavra) for palavra in stopwords_pt]
stopwords_pt = set(stopwords_pt)


def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_pt]
    return tokens


df["tokens_sem_stopwords"] = df["texto_limpo"].apply(remover_stopwords)
df["texto_sem_stopwords"] = df["tokens_sem_stopwords"].str.join(" ")

df[["texto_limpo", "tokens_sem_stopwords", "texto_sem_stopwords"]].head()

,texto_limpo,tokens_sem_stopwords,texto_sem_stopwords
0,o santos encaminhou a extensao de contrato do ...,"[santos, encaminhou, extensao, contrato, ataca...",santos encaminhou extensao contrato atacante r...
1,o cruzeiro anunciou nesta terca feira 14 a ren...,"[cruzeiro, anunciou, nesta, terca, feira, 14, ...",cruzeiro anunciou nesta terca feira 14 renovac...
2,um episodio lamentavel de racismo marcou o cla...,"[episodio, lamentavel, racismo, marcou, classi...",episodio lamentavel racismo marcou classico co...
3,pedro junio vice presidente do cruzeiro explic...,"[pedro, junio, vice, presidente, cruzeiro, exp...",pedro junio vice presidente cruzeiro explicou ...
4,apos a vitoria do flamengo sobre o cruzeiro po...,"[apos, vitoria, flamengo, sobre, cruzeiro, 2, ...",apos vitoria flamengo sobre cruzeiro 2 0 quint...


## 3. Bag of Words

No Bag of Words, cada linha representa uma noticia e cada coluna representa uma palavra. O valor indica quantas vezes aquela palavra apareceu na noticia.

Exemplo: `"senador falou senador"` teria `senador = 2` e `falou = 1`.

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
matriz_bow = vectorizer.fit_transform(df["texto_sem_stopwords"])

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df_bow.head()

,00,000,01,0100,01a,02,03,031,034,038,...,zozimo,zrndj7qcoo,zubeldia,zudcuxlcuk,zugvs0nt0tg,zulu,zuniga,zvezda,zvglllrzo1,zygluzo8cn0
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Removendo colunas com numeros

Agora vamos remover qualquer coluna cujo nome tenha pelo menos um numero.

In [11]:
colunas_com_numeros = [col for col in df_bow.columns if any(char.isdigit() for char in col)]

df_bow = df_bow.drop(columns=colunas_com_numeros)

print(f"{len(colunas_com_numeros)} colunas removidas")
df_bow.head()

1266 colunas removidas


,aa,aab,aazgrqmtyf,abafa,abafar,abaixa,abaixar,abaixo,abaixou,abala,...,zona,zonas,zortea,zorz,zozimo,zubeldia,zudcuxlcuk,zulu,zuniga,zvezda
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,3,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Criando o DataFrame final

Agora vamos juntar os metadados das noticias com as colunas do Bag of Words. Para evitar conflito de nomes, as colunas do Bag of Words recebem o prefixo `bow_`.

In [13]:
metadados = df[["data_publicacao", "titulo", "categoria", "url", "imagem", "subtitulo"]].reset_index(drop=True)
bow_com_prefixo = df_bow.add_prefix("bow_").reset_index(drop=True)

df_final = pd.concat([metadados, bow_com_prefixo], axis=1)

df_final.head()

,data_publicacao,titulo,categoria,url,imagem,subtitulo,bow_aa,bow_aab,bow_aazgrqmtyf,bow_abafa,...,bow_zona,bow_zonas,bow_zortea,bow_zorz,bow_zozimo,bow_zubeldia,bow_zudcuxlcuk,bow_zulu,bow_zuniga,bow_zvezda
0,2026-04-14T16:18:41-03:00,Santos acerta renovação de Robinho Jr. até o f...,Santos,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Clube da Baixada Santista define condições de ...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2026-04-14T15:52:24-03:00,Cruzeiro anuncia renovação com Artur Jorge apó...,Cruzeiro,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Treinador iniciou no clube celeste em 22 de ma...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2026-04-13T18:37:05-03:00,Ex-Corinthians cita caso vivido em clube e cri...,Corinthians,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Betão comentou o episódio recente no Dérbi pau...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2026-04-10T09:00:53-03:00,Cruzeiro: Pedro Junio explica escolha por Artu...,Cruzeiro,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,"Ao CNN Esportes S/A deste domingo (12), vice-p...",0,0,0,0,...,3,0,0,0,0,0,0,0,0,0
4,2026-03-12T08:46:42-03:00,"""Eu sou 200% Flamengo"", diz Leonardo Jardim ap...",Flamengo,https://www.cnnbrasil.com.br/esportes/futebol/...,https://admin.cnnbrasil.com.br/wp-content/uplo...,Treinador destaca evolução da equipe e comenta...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Exemplo de analise

Agora podemos fazer uma analise simples das palavras do Bag of Words: quais aparecem mais, quais aparecem menos e quantas palavras diferentes temos no vocabulario.

In [16]:
colunas_bow = [coluna for coluna in df_final.columns if coluna.startswith("bow_")]

frequencia_palavras = df_final[colunas_bow].sum().sort_values(ascending=False)

print(f"Total de palavras diferentes: {len(frequencia_palavras)}")

frequencia_palavras.head(10)

Total de palavras diferentes: 19136


bow_clube          2183
bow_apos           1864
bow_jogo           1852
bow_corinthians    1740
bow_campeonato     1699
bow_feira          1653
bow_contra         1618
bow_copa           1564
bow_paulo          1550
bow_flamengo       1545
dtype: int64

In [17]:
frequencia_palavras.tail(10)

bow_moy         1
bow_movistar    1
bow_zenon       1
bow_abencoa     1
bow_abeencou    1
bow_abecat      1
bow_abandono    1
bow_zegarra     1
bow_zarate      1
bow_zomba       1
dtype: int64

In [18]:
documentos_por_palavra = (df_final[colunas_bow] > 0).sum().sort_values(ascending=False)
documentos_por_palavra.index = documentos_por_palavra.index.str.replace("bow_", "", regex=False)

documentos_por_palavra.head(10)

feira         1147
apos          1060
campeonato    1031
jogo           958
contra         955
brasileiro     943
clube          935
tecnico        824
nesta          813
ainda          800
dtype: int64

In [21]:
df_final["palavras_unicas"] = (df_final[colunas_bow] > 0).sum(axis=1)

df_final[["titulo", "palavras_unicas"]].sort_values("palavras_unicas", ascending=True).head(10)

,titulo,palavras_unicas
577,Lucas marca para o São Paulo contra o Braganti...,27
1061,Noroeste x Velo Clube: horário e onde assistir...,30
582,São Paulo reclama de pênalti contra o Braganti...,35
1157,Corinthians anuncia ex-goleiro como novo geren...,39
674,Campeonato Carioca 2026: veja os times que dis...,40
581,Bobadilla marca para o São Paulo contra o Brag...,41
555,Palmeiras x Corinthians na semifinal? Veja cen...,41
1340,Veja gol de Igor Thiago para o Brasil na vitór...,42
1437,Flamengo: Bap diz que não vai construir estádio,42
987,Corinthians e Angileri voltam a conversar por ...,42
